# Meta Model

## Problem Definition

**Question.** Given an OOF primary direction, should the strategy act, and at what confidence-derived size?

**Role in the workflow.** Select and tune the secondary act/pass model without changing primary direction.

**Inputs.** Cleaned, weighted events, primary OOF/holdout predictions, primary features, `primary_side`, and `primary_confidence`. `target_return` remains labeling metadata and is not a model input.

**Outputs.** Meta model artifact, candidate/tuning/importance tables, and OOF plus holdout act/probability predictions.

**Why this method.** F1 is primary because the positive class means taking a potentially profitable proposed trade; log loss and precision remain visible.

**Assumptions.** Development meta-labels use only primary OOF predictions; primary direction is immutable; holdout is evaluated once.

**Handoff.** Act/pass and meta probability to `bet_sizing.ipynb`.


## OOF-Only Meta-Label Construction

For development, `meta_label = 1` only when `primary_side × raw_return > 0`. The helper rejects any primary row not explicitly marked `oof`; in-sample primary predictions cannot create meta-labels.


In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, log_loss, precision_score

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.strategy_modeling.cross_validation import PurgedKFold
from src.strategy_modeling.feature_importance import get_orthogonal_features
from src.strategy_modeling.model_workflow import (
    build_candidate_classifiers,
    build_meta_training_frame,
    candidate_parameter_grids,
    generate_oof_predictions,
    get_primary_feature_columns,
)

RANDOM_STATE = 42
period = "2025-01-01_2025-12-31"
event_path = PROJECT_ROOT / f"data/research_data/events/aapl_news_modeling_prepared_{period}.parquet"
artifact_dir = PROJECT_ROOT / "data/model_artifact"

events = pd.read_parquet(event_path).set_index("event_start").sort_index()
primary_predictions = pd.read_parquet(artifact_dir / "primary_predictions.parquet").sort_index()
primary_features = get_primary_feature_columns(events.reset_index())

development_primary = primary_predictions[primary_predictions["partition"].eq("development")]
development_events = events.loc[development_primary.index]
primary_oof_contract = development_primary.rename(columns={"primary_side": "prediction", "primary_probability": "probability"})
meta_development = build_meta_training_frame(
    development_events,
    primary_oof_contract[["prediction", "probability", "prediction_source"]],
)

meta_features = [*primary_features, "primary_side", "primary_confidence"]
X_development = meta_development[meta_features]
y_development = meta_development["meta_label"].astype("int8")
w_development = meta_development["sample_weight"].astype(float)
t1_development = meta_development["event_end"]
cv = PurgedKFold(n_splits=5, t1=t1_development, pct_embargo=0.01)

def score_meta(predictions):
    return {
        "f1": f1_score(y_development, predictions["prediction"], sample_weight=w_development, zero_division=0),
        "log_loss": log_loss(y_development, np.column_stack([1.0 - predictions["probability"], predictions["probability"]]), labels=[0, 1], sample_weight=w_development),
        "precision": precision_score(y_development, predictions["prediction"], sample_weight=w_development, zero_division=0),
    }


candidates = build_candidate_classifiers(random_state=RANDOM_STATE, n_jobs=1)
comparison_rows = []
for name, estimator in candidates.items():
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    comparison_rows.append({"candidate": name, **score_meta(predictions)})

comparison = pd.DataFrame(comparison_rows).set_index("candidate").sort_values(["f1", "log_loss"], ascending=[False, True])
selected_name = comparison.index[0]
display(pd.Series({"meta_events": len(meta_development), "act_labels": int(y_development.sum()), "pass_labels": int((1 - y_development).sum()), "meta_features": len(meta_features)}, name="value").to_frame())
display(comparison)


## Purged Tuning and Meta OOF Output

The selected family is tuned by development F1, with log loss and precision reported. Each stored development meta prediction is out-of-fold under the same purging and embargo contract.


In [ ]:
tuning_rows = []
tuned_oof = {}
for configuration in candidate_parameter_grids()[selected_name]:
    estimator = clone(candidates[selected_name]).set_params(**configuration)
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    key = repr(configuration)
    tuned_oof[key] = predictions
    tuning_rows.append({"configuration": key, **configuration, **score_meta(predictions)})

tuning = pd.DataFrame(tuning_rows).sort_values(["f1", "log_loss"], ascending=[False, True], ignore_index=True)
best_configuration = {key: tuning.loc[0, key] for key in candidate_parameter_grids()[selected_name][0]}
tuned_estimator = clone(candidates[selected_name]).set_params(**best_configuration)
meta_oof = tuned_oof[tuning.loc[0, "configuration"]]

meta_oof_output = meta_development[["event_end", "raw_return", "direction_label", "sample_weight", "primary_side", "primary_probability", "primary_confidence", "meta_label"]].copy()
meta_oof_output["partition"] = "development"
meta_oof_output["meta_action"] = meta_oof["prediction"].astype("int8")
meta_oof_output["meta_probability"] = meta_oof["probability"]
meta_oof_output["prediction_source"] = meta_oof["prediction_source"]
meta_oof_output["cv_fold"] = meta_oof["fold"]
display(tuning)
display(meta_oof_output.head())


## Development Feature Importance and Error Analysis

The same MDI, MDA, SFI, and orthogonal views are recomputed for the meta target on development only. This distinguishes features useful for direction from features useful for deciding whether to trust that direction.


In [ ]:
np.random.seed(RANDOM_STATE)
diagnostic_forest = RandomForestClassifier(
    n_estimators=120,
    class_weight="balanced_subsample",
    max_features="sqrt",
    n_jobs=1,
    random_state=RANDOM_STATE,
).fit(X_development, y_development, sample_weight=w_development)

mdi = pd.Series(diagnostic_forest.feature_importances_, index=meta_features, name="mdi")
mda_result = permutation_importance(
    diagnostic_forest,
    X_development,
    y_development,
    scoring="neg_log_loss",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
    sample_weight=w_development,
)
mda = pd.Series(mda_result.importances_mean, index=meta_features, name="mda")

sfi_scores = {}
for feature in meta_features:
    single_predictions = generate_oof_predictions(candidates["random_forest"], X_development[[feature]], y_development, w_development, cv, positive_label=1)
    sfi_scores[feature] = -score_meta(single_predictions)["log_loss"]
sfi = pd.Series(sfi_scores, name="sfi_neg_log_loss")

importance = pd.concat([mdi, mda, sfi], axis=1).sort_values("mda", ascending=False)
orthogonal = get_orthogonal_features(X_development, var_thres=0.95)
orthogonal_summary = pd.DataFrame({"component": orthogonal.columns, "label_correlation": [orthogonal[column].corr(y_development) for column in orthogonal.columns]})

importance.to_parquet(artifact_dir / "meta_feature_importance.parquet")
orthogonal_summary.to_parquet(artifact_dir / "meta_orthogonal_features.parquet", index=False)
display(importance.head(10))
display(orthogonal_summary.head())


## Final Fit and One-Time Holdout Evaluation

The final meta estimator fits all OOF-derived development labels. Holdout inputs use predictions from the already frozen primary model; holdout outcomes are revealed only to report final metrics.


In [ ]:
final_meta = clone(tuned_estimator).fit(X_development, y_development, sample_weight=w_development.to_numpy())

holdout_primary = primary_predictions[primary_predictions["partition"].eq("holdout")]
holdout_events = events.loc[holdout_primary.index].copy()
holdout_events["primary_side"] = holdout_primary["primary_side"].astype("int8")
holdout_events["primary_probability"] = holdout_primary["primary_probability"]
holdout_events["primary_confidence"] = holdout_primary["primary_confidence"]
holdout_events["meta_label"] = (holdout_events["primary_side"] * holdout_events["raw_return"] > 0).astype("int8")

holdout_probability = final_meta.predict_proba(holdout_events[meta_features])[:, list(final_meta.classes_).index(1)]
holdout_action = final_meta.predict(holdout_events[meta_features]).astype("int8")

meta_holdout_output = holdout_events[["event_end", "raw_return", "direction_label", "sample_weight", "primary_side", "primary_probability", "primary_confidence", "meta_label"]].copy()
meta_holdout_output["partition"] = "holdout"
meta_holdout_output["meta_action"] = holdout_action
meta_holdout_output["meta_probability"] = holdout_probability
meta_holdout_output["prediction_source"] = "holdout"
meta_holdout_output["cv_fold"] = pd.NA

holdout_metrics = pd.Series(
    {
        "f1": f1_score(holdout_events["meta_label"], holdout_action, sample_weight=holdout_events["sample_weight"], zero_division=0),
        "log_loss": log_loss(holdout_events["meta_label"], np.column_stack([1.0 - holdout_probability, holdout_probability]), labels=[0, 1], sample_weight=holdout_events["sample_weight"]),
        "precision": precision_score(holdout_events["meta_label"], holdout_action, sample_weight=holdout_events["sample_weight"], zero_division=0),
    },
    name="holdout",
)

meta_predictions = pd.concat([meta_oof_output, meta_holdout_output]).sort_index()
tuning_artifact = tuning.copy()
for column in tuning_artifact.columns:
    if column.startswith("model__") and tuning_artifact[column].dtype == "object":
        tuning_artifact[column] = tuning_artifact[column].astype(str)
meta_predictions.to_parquet(artifact_dir / "meta_predictions.parquet")
comparison.to_parquet(artifact_dir / "meta_candidate_metrics.parquet")
tuning_artifact.to_parquet(artifact_dir / "meta_tuning_metrics.parquet", index=False)
holdout_metrics.to_frame().to_parquet(artifact_dir / "meta_holdout_metrics.parquet")
joblib.dump(
    {
        "estimator": final_meta,
        "feature_columns": meta_features,
        "selected_candidate": selected_name,
        "best_configuration": best_configuration,
        "random_state": RANDOM_STATE,
    },
    artifact_dir / "meta_model.joblib",
)

display(holdout_metrics.to_frame())
print(artifact_dir / "meta_model.joblib")


## Results, Limitations, and Handoff

A meta model may fail to improve primary-only results; that is a valid finding, not a trigger to retune on holdout. The action threshold remains the fitted classifier decision and direction always remains `primary_side`.

The next notebook receives fixed act/pass decisions and meta probabilities. No conclusion in this notebook is evidence of live-trading profitability.
